# 🚗 Multi-Camera Vehicle Tracking & Traffic Counting with ByteTrack
### Automated Video Surveillance and Public Safety Pipeline

**Reference Repository:** [aatansen/Violence-Detection-Using-YOLOv8-Towards-Automated-Video-Surveillance-and-Public-Safety](https://github.com/aatansen/Violence-Detection-Using-YOLOv8-Towards-Automated-Video-Surveillance-and-Public-Safety)

This notebook demonstrates multi-camera vehicle tracking using **YOLOv8s**, **ByteTrack**, and **Supervision** based on **Update 03** from the reference project.

## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics supervision opencv-python matplotlib

## 2. Initialize ByteTrack & Supervision Line Counter

In [ ]:
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO

# Load trained YOLOv8 model for vehicle detection
model = YOLO('yolov8s.pt')

# Initialize ByteTrack Tracker
byte_tracker = sv.ByteTrack()

# Define Virtual Counting Line (x1, y1) to (x2, y2)
LINE_START = sv.Point(50, 360)
LINE_END = sv.Point(1230, 360)

line_zone = sv.LineZone(start=LINE_START, end=LINE_END)
line_zone_annotator = sv.LineZoneAnnotator(thickness=2, text_thickness=2, text_scale=0.8)
box_annotator = sv.BoxAnnotator(thickness=2, text_thickness=1, text_scale=0.5)

print("ByteTrack Vehicle Counter Initialized Successfully!")

## 3. Frame Processing & Vehicle Counting Loop

In [ ]:
def process_frame(frame: np.ndarray) -> np.ndarray:
    results = model(frame, conf=0.25)[0]
    detections = sv.Detections.from_ultralytics(results)
    
    # Filter vehicles (car=2, motorcycle=3, bus=5, truck=7 in COCO dataset)
    vehicle_classes = [2, 3, 5, 7]
    detections = detections[np.isin(detections.class_id, vehicle_classes)]
    
    # Update Tracker
    detections = byte_tracker.update_with_detections(detections=detections)
    
    # Update Line Counting Zone
    line_zone.trigger(detections=detections)
    
    # Annotate Frame
    labels = [f"#{tracker_id} {model.model.names[class_id]}" for _, _, _, class_id, tracker_id in detections]
    frame = box_annotator.annotate(scene=frame, detections=detections, labels=labels)
    frame = line_zone_annotator.annotate(state=frame, line_counter=line_zone)
    return frame